# Pixel shuffle with view and permute

Super-resolution networks upsample by predicting extra channels and then rearranging them into space. A conv outputs `C * r * r` channels at low resolution; **pixel shuffle** turns every group of $r^2$ channels into an $r \times r$ block of pixels.

Write `pixel_shuffle(x, r)`:

- `x`: tensor `(B, C * r * r, H, W)`, any float dtype.
- `r`: positive int, the upscale factor. The channel count is always divisible by $r^2$.
- Return a tensor `(B, C, H * r, W * r)` with

$$\text{out}[b,\ c,\ h r + i,\ w r + j] = x[b,\ c r^2 + i r + j,\ h,\ w], \qquad 0 \le i, j < r.$$

So within each group of $r^2$ channels, the channel offset $i r + j$ picks the row $i$ and column $j$ inside the block. This is the same layout as `torch.nn.functional.pixel_shuffle`, but don't call it: use `reshape`/`view`, `permute` and `reshape` again.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def pixel_shuffle(x, r):
    """x: (B, C*r*r, H, W). Return (B, C, H*r, W*r)."""
    B, Cr2, H, W = x.shape
    return x.reshape(B, Cr2 // (r * r), H * r, W * r)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0]], [[1.0]], [[2.0]], [[3.0]]]], "dtype": "float64"}, 2], "e": {"__nd__": [[[[0.0, 1.0], [2.0, 3.0]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[0.0, 1.0]], [[2.0, 3.0]], [[4.0, 5.0]], [[6.0, 7.0]]]], "dtype": "float64"}, 2], "e": {"__nd__": [[[[0.0, 2.0, 1.0, 3.0], [4.0, 6.0, 5.0, 7.0]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[0.0012301533574825742, 0.2987455375084699, -0.2741378553622176], [-0.8905918387572742, -0.45467078517172255, -0.9916465549964624]], [[0.060143602597438485, 1.3402152455545335, -0.49220651855132963], [-0.6204748998199404, 0.4898420501851982, 0.35688700816006075]], [[0.10541424899789856, -0.9304680447082047, -0.02925182246327349], [0.6953031944582878, -1.344214547285082, -0.45761576104021817]], [[-1.901222739800844, -1.289537739784976, -1.8417350377917323], [-0.23509113107468127, -1.2674464814437032, 0.2712643588217015]], [[0.15675108662422516, -0.18693094462995438, -2.516759710820513], [-0.5386928958466366, -0.048500945401071985, 0.11330898600330756]], [[-1.5301357655053935, -0.47775327603393064, -0.9785190780566395], [-0.8088372394255993, 1.0608986233860787, -0.8075346753318965]], [[-0.0325217049455206, 0.8843898673831739, -0.583600432743302], [-0.11170194958415963, 0.11046414324948059, 0.06378177425506196]], [[-1.2250558264176934, 0.0761402303770081, 1.3588234217415376], [-1.5471446781284823, 0.8593826880215982, 0.11935402569658124]], [[-0.6414703941072214, 2.000416546342423, 0.7622597120847118], [-1.1992889021052233, 0.07451622877146342, 0.5766895836701853]]], [[[-0.1887821253507493, 0.682910267195206, -0.06651732014941557], [0.6672475608343279, 1.438522591656152, -0.6756622510056528]], [[0.20313861038960904, -0.46330757653841514, 0.12726841122583082], [-1.18719452785014, -0.5793015965026732, -0.1961959728044967]], [[0.8987638721004078, 1.145222007454132, -1.323527792484255], [-0.7946423659870495, 0.6469034225734218, -1.9924197841744944]], [[-0.46316986495236695, -0.09728692567008902, 1.2570149772868198], [0.6894039005707556, -0.32721342022219785, -0.3685758940999591]], [[-0.25019540051792494, 1.5235294004561601, -0.4280249425728672], [-0.3036803883647294, 0.35258906728526535, -0.12077044508645512]], [[-0.19728422796572256, -1.1140671431510563, -0.011521468038548173], [-0.4435812229744192, 1.1661277761902227, 0.6530885027011638]], [[-0.024143613009932233, 0.6683810232673438, -0.3398695517131494], [1.052126358426947, -0.005399560671626605, 0.5833823541804138]], [[-1.2908932453234871, 0.34668004887842974, -1.6882041173665416], [-2.0353289449399323, -0.3044768777114372, -0.8999276075985952]], [[0.16405279571222256, 2.2447566264860495, -0.8317231814120817], [-0.6239435864439059, 0.2054039460646989, 0.49301329141235634]]]], "dtype": "float64"}, 3], "e": {"__nd__": [[[[0.0012301533574825742, 0.060143602597438485, 0.10541424899789856, 0.2987455375084699, 1.3402152455545335, -0.9304680447082047, -0.2741378553622176, -0.49220651855132963, -0.02925182246327349], [-1.901222739800844, 0.15675108662422516, -1.5301357655053935, -1.289537739784976, -0.18693094462995438, -0.47775327603393064, -1.8417350377917323, -2.516759710820513, -0.9785190780566395], [-0.0325217049455206, -1.2250558264176934, -0.6414703941072214, 0.8843898673831739, 0.0761402303770081, 2.000416546342423, -0.583600432743302, 1.3588234217415376, 0.7622597120847118], [-0.8905918387572742, -0.6204748998199404, 0.6953031944582878, -0.45467078517172255, 0.4898420501851982, -1.344214547285082, -0.9916465549964624, 0.35688700816006075, -0.45761576104021817], [-0.23509113107468127, -0.5386928958466366, -0.8088372394255993, -1.2674464814437032, -0.048500945401071985, 1.0608986233860787, 0.2712643588217015, 0.11330898600330756, -0.8075346753318965], [-0.11170194958415963, -1.5471446781284823, -1.1992889021052233, 0.11046414324948059, 0.8593826880215982, 0.07451622877146342, 0.06378177425506196, 0.11935402569658124, 0.5766895836701853]]], [[[-0.1887821253507493, 0.20313861038960904, 0.8987638721004078, 0.682910267195206, -0.46330757653841514, 1.145222007454132, -0.06651732014941557, 0.12726841122583082, -1.323527792484255], [-0.46316986495236695, -0.25019540051792494, -0.19728422796572256, -0.09728692567008902, 1.5235294004561601, -1.1140671431510563, 1.2570149772868198, -0.4280249425728672, -0.011521468038548173], [-0.024143613009932233, -1.2908932453234871, 0.16405279571222256, 0.6683810232673438, 0.34668004887842974, 2.2447566264860495, -0.3398695517131494, -1.6882041173665416, -0.8317231814120817], [0.6672475608343279, -1.18719452785014, -0.7946423659870495, 1.438522591656152, -0.5793015965026732, 0.6469034225734218, -0.6756622510056528, -0.1961959728044967, -1.9924197841744944], [0.6894039005707556, -0.3036803883647294, -0.4435812229744192, -0.32721342022219785, 0.35258906728526535, 1.1661277761902227, -0.3685758940999591, -0.12077044508645512, 0.6530885027011638], [1.052126358426947, -2.0353289449399323, -0.6239435864439059, -0.005399560671626605, -0.3044768777114372, 0.2054039460646989, 0.5833823541804138, -0.8999276075985952, 0.49301329141235634]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-0.1764060659057582, -0.20593033025321647, 0.7024629551205442, 0.5199076370338984], [-1.0336758320736887, -0.07918131861584184, 0.035286848661474135, -1.0544846220491104], [0.25983910067436333, -0.8579564771765439, 0.9720667079170427, 0.1927459126050724], [0.08930648576905029, -0.591028352856274, -0.11860982387769403, -1.9977462929070549]], [[-1.1314074705230586, 0.3628397991887543, -2.1285670418221447, 0.8466085214811634], [-1.7460964753739088, 0.7567385026642676, -0.8454970328793241, 0.7789910843424612], [0.1309512075847998, -1.5368349402914887, 1.2491487495584548, 1.4417071555226115], [-0.0658049060002071, -0.27391627217232034, -0.15986696597063635, -0.9751523227787462]], [[1.0985867597569177, -0.5428919317301868, -0.051190412691676575, -0.7932964032030436], [-0.6260730997201972, -1.2777251516511705, 1.2570693137143927, -0.15408757320601318], [0.9659216187288089, 0.01332459691325976, -0.6944035277028942, -0.3266852600022522], [-0.5602310505028996, 0.007959099184913658, -0.3752668396769557, -0.29992171594179834]]]], "dtype": "float64"}, 1], "e": {"__nd__": [[[[-0.1764060659057582, -0.20593033025321647, 0.7024629551205442, 0.5199076370338984], [-1.0336758320736887, -0.07918131861584184, 0.035286848661474135, -1.0544846220491104], [0.25983910067436333, -0.8579564771765439, 0.9720667079170427, 0.1927459126050724], [0.08930648576905029, -0.591028352856274, -0.11860982387769403, -1.9977462929070549]], [[-1.1314074705230586, 0.3628397991887543, -2.1285670418221447, 0.8466085214811634], [-1.7460964753739088, 0.7567385026642676, -0.8454970328793241, 0.7789910843424612], [0.1309512075847998, -1.5368349402914887, 1.2491487495584548, 1.4417071555226115], [-0.0658049060002071, -0.27391627217232034, -0.15986696597063635, -0.9751523227787462]], [[1.0985867597569177, -0.5428919317301868, -0.051190412691676575, -0.7932964032030436], [-0.6260730997201972, -1.2777251516511705, 1.2570693137143927, -0.15408757320601318], [0.9659216187288089, 0.01332459691325976, -0.6944035277028942, -0.3266852600022522], [-0.5602310505028996, 0.007959099184913658, -0.3752668396769557, -0.29992171594179834]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-1.3785746841359137, -0.8068459276211205], [1.65405754880043, -0.6712332162517173], [-1.0540937876234928, 0.33732633257503086]], [[1.407272199556532, -1.454024302481208], [-0.20852184825127065, -0.6320525539349078], [-1.7610194743274028, 0.7349267092583915]], [[-0.02344391855484465, 0.07144184429260535], [-0.7523114724455892, 0.4547841629969166], [-0.539297349487321, -0.1429032084967607]], [[-1.108260792181513, -1.2161027602081953], [1.3355318553000226, -0.5071047045384752], [0.2916803563558019, -0.03379043476737232]], [[-0.44114520276218416, -0.5079609703372195], [0.6300825914455861, -0.3018676045339098], [-0.15144364817129202, 0.022221557194479512]], [[1.1765083202520015, 0.6805109746331383], [0.3826002677279928, -0.5635713933532417], [-1.3819687200064654, 0.949529931735302]], [[0.9664471342208761, -0.1407083991233403], [0.541883841603518, 0.781443053885405], [0.8311844526649818, 0.9213834732464707]], [[-0.45561765066113274, 1.5149729826647176], [-1.2465930582857083, 0.861723076993649], [0.4939320790307585, 0.8736191135245024]]], [[[1.8790083070967372, 1.484445473378649], [-1.145176907189325, -1.688671580267857], [0.8168890590537966, -1.015012415875923]], [[-0.0124053886475133, 0.839727477005194], [-1.6437973511855861, -2.1099804543094787], [0.259299020337227, 0.04438588961329704]], [[-0.24580307138192253, 0.03853476377021816], [-0.8605156073672797, -1.5134944072721068], [-0.1666548508803217, -0.9717086910493917]], [[-1.643481322411107, 0.5056810713932767], [-0.061398628422175805, 0.40652853663281385], [-0.9892949414259369, -0.6580587918366578]], [[-0.9990430272201656, -0.8866418670580481], [0.19540791774940214, -0.7829746163349458], [0.3560662888202746, 0.3397559254182162]], [[2.0251609868801026, -1.3927890458668095], [0.8879023778350454, -0.0894879618884183], [-0.01402973013099694, -1.4498638219066111]], [[-0.4601938127246543, 0.743197263443954], [-0.08247837201730518, 0.08105437041126562], [-0.29071668060054134, 1.1545697469862868]], [[-0.021472567043494796, -2.200415672482062], [-0.6920725504744906, -1.968796607080243], [-3.2514384154965383, -0.5301153497723121]]], [[[1.3335598501027237, 0.04711990613059292], [-1.1725457074049794, -0.9406998682024224], [1.1306132302500087, 0.15762662339846478]], [[0.04799924156205696, -0.05346178883805718], [0.038400261555346496, 0.8054056469437983], [0.5525672973560755, 0.21570470002449457]], [[-1.0428683575900106, 0.511108764909727], [-0.6842470779924941, 1.0938456759004787], [-1.2710508217241274, -0.13762097627558853]], [[-0.007358286291270949, -1.3246455506441366], [1.721971643856479, 1.4604067672595522], [-0.46358376160908316, 0.7717211655645232]], [[0.37867606967021183, -2.613559463345258], [0.2503980162775905, -0.061344079833248494], [0.08321735347453037, -1.0768749198271783]], [[-0.2693470462204049, -0.17825876338229713], [1.1880942172123188, 0.33442706039101433], [-0.005555030202663769, 1.5289699979109401]], [[-0.5552479297636472, -0.3894303048753847], [-1.8167550113390598, 1.5691058462836123], [0.9643323681944285, 0.9168480445117076]], [[0.6688984464469374, 0.110148587663652], [0.21548893918232892, -0.2520065902673078], [-0.20360042572294623, 0.05430361171369427]]]], "dtype": "float64"}, 2], "e": {"__nd__": [[[[-1.3785746841359137, 1.407272199556532, -0.8068459276211205, -1.454024302481208], [-0.02344391855484465, -1.108260792181513, 0.07144184429260535, -1.2161027602081953], [1.65405754880043, -0.20852184825127065, -0.6712332162517173, -0.6320525539349078], [-0.7523114724455892, 1.3355318553000226, 0.4547841629969166, -0.5071047045384752], [-1.0540937876234928, -1.7610194743274028, 0.33732633257503086, 0.7349267092583915], [-0.539297349487321, 0.2916803563558019, -0.1429032084967607, -0.03379043476737232]], [[-0.44114520276218416, 1.1765083202520015, -0.5079609703372195, 0.6805109746331383], [0.9664471342208761, -0.45561765066113274, -0.1407083991233403, 1.5149729826647176], [0.6300825914455861, 0.3826002677279928, -0.3018676045339098, -0.5635713933532417], [0.541883841603518, -1.2465930582857083, 0.781443053885405, 0.861723076993649], [-0.15144364817129202, -1.3819687200064654, 0.022221557194479512, 0.949529931735302], [0.8311844526649818, 0.4939320790307585, 0.9213834732464707, 0.8736191135245024]]], [[[1.8790083070967372, -0.0124053886475133, 1.484445473378649, 0.839727477005194], [-0.24580307138192253, -1.643481322411107, 0.03853476377021816, 0.5056810713932767], [-1.145176907189325, -1.6437973511855861, -1.688671580267857, -2.1099804543094787], [-0.8605156073672797, -0.061398628422175805, -1.5134944072721068, 0.40652853663281385], [0.8168890590537966, 0.259299020337227, -1.015012415875923, 0.04438588961329704], [-0.1666548508803217, -0.9892949414259369, -0.9717086910493917, -0.6580587918366578]], [[-0.9990430272201656, 2.0251609868801026, -0.8866418670580481, -1.3927890458668095], [-0.4601938127246543, -0.021472567043494796, 0.743197263443954, -2.200415672482062], [0.19540791774940214, 0.8879023778350454, -0.7829746163349458, -0.0894879618884183], [-0.08247837201730518, -0.6920725504744906, 0.08105437041126562, -1.968796607080243], [0.3560662888202746, -0.01402973013099694, 0.3397559254182162, -1.4498638219066111], [-0.29071668060054134, -3.2514384154965383, 1.1545697469862868, -0.5301153497723121]]], [[[1.3335598501027237, 0.04799924156205696, 0.04711990613059292, -0.05346178883805718], [-1.0428683575900106, -0.007358286291270949, 0.511108764909727, -1.3246455506441366], [-1.1725457074049794, 0.038400261555346496, -0.9406998682024224, 0.8054056469437983], [-0.6842470779924941, 1.721971643856479, 1.0938456759004787, 1.4604067672595522], [1.1306132302500087, 0.5525672973560755, 0.15762662339846478, 0.21570470002449457], [-1.2710508217241274, -0.46358376160908316, -0.13762097627558853, 0.7717211655645232]], [[0.37867606967021183, -0.2693470462204049, -2.613559463345258, -0.17825876338229713], [-0.5552479297636472, 0.6688984464469374, -0.3894303048753847, 0.110148587663652], [0.2503980162775905, 1.1880942172123188, -0.061344079833248494, 0.33442706039101433], [-1.8167550113390598, 0.21548893918232892, 1.5691058462836123, -0.2520065902673078], [0.08321735347453037, -0.005555030202663769, -1.0768749198271783, 1.5289699979109401], [0.9643323681944285, -0.20360042572294623, 0.9168480445117076, 0.05430361171369427]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[0.0, 1.0], [2.0, 3.0]], [[4.0, 5.0], [6.0, 7.0]], [[8.0, 9.0], [10.0, 11.0]], [[12.0, 13.0], [14.0, 15.0]], [[16.0, 17.0], [18.0, 19.0]], [[20.0, 21.0], [22.0, 23.0]], [[24.0, 25.0], [26.0, 27.0]], [[28.0, 29.0], [30.0, 31.0]], [[32.0, 33.0], [34.0, 35.0]], [[36.0, 37.0], [38.0, 39.0]], [[40.0, 41.0], [42.0, 43.0]], [[44.0, 45.0], [46.0, 47.0]], [[48.0, 49.0], [50.0, 51.0]], [[52.0, 53.0], [54.0, 55.0]], [[56.0, 57.0], [58.0, 59.0]], [[60.0, 61.0], [62.0, 63.0]]], [[[64.0, 65.0], [66.0, 67.0]], [[68.0, 69.0], [70.0, 71.0]], [[72.0, 73.0], [74.0, 75.0]], [[76.0, 77.0], [78.0, 79.0]], [[80.0, 81.0], [82.0, 83.0]], [[84.0, 85.0], [86.0, 87.0]], [[88.0, 89.0], [90.0, 91.0]], [[92.0, 93.0], [94.0, 95.0]], [[96.0, 97.0], [98.0, 99.0]], [[100.0, 101.0], [102.0, 103.0]], [[104.0, 105.0], [106.0, 107.0]], [[108.0, 109.0], [110.0, 111.0]], [[112.0, 113.0], [114.0, 115.0]], [[116.0, 117.0], [118.0, 119.0]], [[120.0, 121.0], [122.0, 123.0]], [[124.0, 125.0], [126.0, 127.0]]]], "dtype": "float64"}, 4], "e": {"__nd__": [[[[0.0, 4.0, 8.0, 12.0, 1.0, 5.0, 9.0, 13.0], [16.0, 20.0, 24.0, 28.0, 17.0, 21.0, 25.0, 29.0], [32.0, 36.0, 40.0, 44.0, 33.0, 37.0, 41.0, 45.0], [48.0, 52.0, 56.0, 60.0, 49.0, 53.0, 57.0, 61.0], [2.0, 6.0, 10.0, 14.0, 3.0, 7.0, 11.0, 15.0], [18.0, 22.0, 26.0, 30.0, 19.0, 23.0, 27.0, 31.0], [34.0, 38.0, 42.0, 46.0, 35.0, 39.0, 43.0, 47.0], [50.0, 54.0, 58.0, 62.0, 51.0, 55.0, 59.0, 63.0]]], [[[64.0, 68.0, 72.0, 76.0, 65.0, 69.0, 73.0, 77.0], [80.0, 84.0, 88.0, 92.0, 81.0, 85.0, 89.0, 93.0], [96.0, 100.0, 104.0, 108.0, 97.0, 101.0, 105.0, 109.0], [112.0, 116.0, 120.0, 124.0, 113.0, 117.0, 121.0, 125.0], [66.0, 70.0, 74.0, 78.0, 67.0, 71.0, 75.0, 79.0], [82.0, 86.0, 90.0, 94.0, 83.0, 87.0, 91.0, 95.0], [98.0, 102.0, 106.0, 110.0, 99.0, 103.0, 107.0, 111.0], [114.0, 118.0, 122.0, 126.0, 115.0, 119.0, 123.0, 127.0]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[1.5118295115637772, 0.5556879018080786, -0.058460125758861455, -0.5793920504433961, -0.6349984874116595]], [[1.6027050111724899, 0.5066873042271038, 0.06755055831065467, -0.34618184215272274, -1.1090534117005277]], [[-0.06686155746989603, 0.873658034470201, -0.39253724683749003, -0.2272425830005601, -0.22103435143897837]], [[0.10959436799816868, -1.593010878745609, -0.235398351965091, -0.8543955181540714, 0.8845850325626053]], [[-0.7705986765648598, 0.5770467473006955, 1.5244374527267601, -0.3135962612816255, -0.6015762521234661]], [[0.1914322591260928, -0.0020290276090286124, -0.9936163473397114, 0.4609194157133795, 2.015516053475408]], [[-0.25811199526572426, -0.2028761960646632, -1.0449320235690858, 0.3190884516865476, -1.2469762247076281]], [[-1.1069310243514836, 1.2796672718437896, -0.9054530593946442, 1.0813575693313382, 1.524358003782807]], [[0.2593264574767428, 0.5533914028508236, 1.9522509436867783, -0.19672840206571665, -0.5930057969416156]], [[-1.3532310647829418, 0.041707080346346355, 1.4791442109183066, 0.9595953814008653, -0.9420912414179574]], [[-0.855375383832596, -0.5041712293409636, 0.2922680824953417, -0.20531142710368377, 0.21445394544308127]], [[0.2967393780454473, -0.2987735866784475, -0.040173941602943734, 0.20659237721603418, -0.08396970353317602]]], [[[0.5035208649940508, 1.8708757733280048, 0.5919722847708305, 0.05581054640855185, -1.6861186233619148]], [[0.38795708601520174, -1.9466784117672074, -1.409034086156122, 0.8546392524687652, 0.7062350601366643]], [[-0.14993899834888477, -1.7100111061522156, -0.37134851689085024, -0.678738318216024, 0.6368407461302101]], [[2.2577305324275314, 0.21693048795465586, -0.7793111280321169, -1.1705515303032201, -0.05609428076827262]], [[-0.17679319351358783, -1.1515191595274716, 0.1163552356921842, -1.1509135213108617, 1.1121072717293008]], [[1.0626506057881553, 1.0847512652964209, -0.4740504706085282, 0.5145205633180888, -0.13206950521935684]], [[-0.38881206140248165, -0.3391456571420661, -1.2997152730341506, -1.4438636958951212, 0.7943147908760457]], [[-0.19123540391952007, 0.21642485112762316, 1.0017103884541199, -1.7331324950160742, -0.784130073815894]], [[0.17533484346507067, 0.39209478666557296, -0.3770744504010968, 1.0291792326978486, 0.2103949388296369]], [[-1.2133821388278598, -0.9307637759906452, 0.8054710604946225, 0.4638314564560598, -1.899061239006601]], [[1.3477123038624088, 0.5980270310938874, 1.3433429876964984, -0.383700881075084, -0.2957062213323259]], [[-1.126457084751319, 2.5369295701318877, -0.17545422540714875, 1.5875331920819455, -0.6472924513872795]]]], "dtype": "float64"}, 2], "e": {"__nd__": [[[[1.5118295115637772, 1.6027050111724899, 0.5556879018080786, 0.5066873042271038, -0.058460125758861455, 0.06755055831065467, -0.5793920504433961, -0.34618184215272274, -0.6349984874116595, -1.1090534117005277], [-0.06686155746989603, 0.10959436799816868, 0.873658034470201, -1.593010878745609, -0.39253724683749003, -0.235398351965091, -0.2272425830005601, -0.8543955181540714, -0.22103435143897837, 0.8845850325626053]], [[-0.7705986765648598, 0.1914322591260928, 0.5770467473006955, -0.0020290276090286124, 1.5244374527267601, -0.9936163473397114, -0.3135962612816255, 0.4609194157133795, -0.6015762521234661, 2.015516053475408], [-0.25811199526572426, -1.1069310243514836, -0.2028761960646632, 1.2796672718437896, -1.0449320235690858, -0.9054530593946442, 0.3190884516865476, 1.0813575693313382, -1.2469762247076281, 1.524358003782807]], [[0.2593264574767428, -1.3532310647829418, 0.5533914028508236, 0.041707080346346355, 1.9522509436867783, 1.4791442109183066, -0.19672840206571665, 0.9595953814008653, -0.5930057969416156, -0.9420912414179574], [-0.855375383832596, 0.2967393780454473, -0.5041712293409636, -0.2987735866784475, 0.2922680824953417, -0.040173941602943734, -0.20531142710368377, 0.20659237721603418, 0.21445394544308127, -0.08396970353317602]]], [[[0.5035208649940508, 0.38795708601520174, 1.8708757733280048, -1.9466784117672074, 0.5919722847708305, -1.409034086156122, 0.05581054640855185, 0.8546392524687652, -1.6861186233619148, 0.7062350601366643], [-0.14993899834888477, 2.2577305324275314, -1.7100111061522156, 0.21693048795465586, -0.37134851689085024, -0.7793111280321169, -0.678738318216024, -1.1705515303032201, 0.6368407461302101, -0.05609428076827262]], [[-0.17679319351358783, 1.0626506057881553, -1.1515191595274716, 1.0847512652964209, 0.1163552356921842, -0.4740504706085282, -1.1509135213108617, 0.5145205633180888, 1.1121072717293008, -0.13206950521935684], [-0.38881206140248165, -0.19123540391952007, -0.3391456571420661, 0.21642485112762316, -1.2997152730341506, 1.0017103884541199, -1.4438636958951212, -1.7331324950160742, 0.7943147908760457, -0.784130073815894]], [[0.17533484346507067, -1.2133821388278598, 0.39209478666557296, -0.9307637759906452, -0.3770744504010968, 0.8054710604946225, 1.0291792326978486, 0.4638314564560598, 0.2103949388296369, -1.899061239006601], [1.3477123038624088, -1.126457084751319, 0.5980270310938874, 2.5369295701318877, 1.3433429876964984, -0.17545422540714875, -0.383700881075084, 1.5875331920819455, -0.2957062213323259, -0.6472924513872795]]]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(pixel_shuffle, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")